# LeetCode 842 — Split Array into Fibonacci Sequence bằng Python

Link đề: https://leetcode.com/problems/split-array-into-fibonacci-sequence/ · Độ khó: **Medium** · Chủ đề: String, Backtracking

Skeleton LeetCode:

```python
class Solution:
    def splitIntoFibonacci(self, num: str) -> List[int]:
        ...
```

Ghi chú: bài có **nhiều đáp án đúng** (đề: "trả về **bất kỳ** dãy Fibonacci-like nào"; ví dụ `"1101111"` cho `[11,0,11,11]` hoặc `[110,1,111]`). Vì vậy notebook **không so đáp án cứng** mà dùng **hàm kiểm tra tính hợp lệ** `is_valid_fib(num, seq)` và so sánh **có/không có đáp án** với oracle vét cạn (liệt kê *mọi* cách tách).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tách chuỗi chữ số `num` thành một dãy **kiểu Fibonacci**: `≥ 3` số nguyên không âm, mỗi số `< 2^31`, mỗi số **không có chữ số 0 thừa ở đầu** (trừ chính số `0`), và `f[i] + f[i+1] == f[i+2]`; trả về một dãy bất kỳ hoặc `[]`.
2. Nắm các công cụ Python: cắt chuỗi `num[i:j]`, `int(...)`, `str.startswith(prefix, pos)`, quay lui (backtracking) với **cắt tỉa**, `itertools.combinations` (chọn vị trí cắt), và viết **hàm kiểm tra tính hợp lệ** cho bài có nhiều đáp án.
3. Đi từ **liệt kê mọi cách cắt rồi kiểm tra (brute force)** → **quay lui có cắt tỉa theo tổng** → **chỉ liệt kê hai số đầu, phần còn lại suy ra** → **dựng chuỗi kỳ vọng rồi so sánh**.
4. Hiểu mấu chốt: dãy hoàn toàn được **xác định bởi hai số đầu** `(f0, f1)`, và mỗi số `< 2^31` có tối đa **10 chữ số**, nên chỉ có `≤ 10 × 10 = 100` cặp ứng viên; mỗi cặp kiểm tra bằng một lượt dựng dãy.
5. Tránh các lỗi kinh điển: bỏ quên điều kiện **32-bit** (Python không tràn), cho phép **số 0 đứng đầu**, quên `len ≥ 3`, không dùng **hết** chuỗi, và cấm nhầm số `0` đơn lẻ.

## 2. Đề bài & ví dụ

Cho chuỗi chữ số `num`, ví dụ `"123456579"`; có thể tách thành dãy kiểu Fibonacci `[123, 456, 579]`.

**Dãy kiểu Fibonacci** `f` là danh sách số nguyên không âm thoả:

- `0 <= f[i] < 2^31` (mỗi số vừa số nguyên có dấu 32-bit);
- `f.length >= 3`;
- `f[i] + f[i + 1] == f[i + 2]` với mọi `0 <= i < f.length − 2`.

Khi tách chuỗi thành các đoạn, mỗi đoạn **không được có chữ số 0 thừa ở đầu**, trừ khi đoạn đó chính là số `0`.

Trả về **bất kỳ** dãy kiểu Fibonacci nào tách được từ `num`, hoặc `[]` nếu không thể.

**Ví dụ 1:** `num = "1101111"` → `[11, 0, 11, 11]` (cũng chấp nhận `[110, 1, 111]`).

**Ví dụ 2:** `num = "112358130"` → `[]` (không thể).

**Ví dụ 3:** `num = "0123"` → `[]` (không được có số 0 thừa ở đầu nên `"01"`, `"2"`, `"3"` không hợp lệ).

**Ràng buộc:** `1 <= num.length <= 200`; `num` chỉ gồm chữ số.

In [1]:
import random
import time
from itertools import combinations, product
from typing import List

LIMIT = 2 ** 31 - 1
ex1, ex2, ex3 = "1101111", "112358130", "0123"
print("2^31 - 1 =", LIMIT, "(", len(str(LIMIT)), "chữ số )")
print("số có 11 chữ số trở lên luôn >= 10^10 > 2^31:", 10 ** 10 > LIMIT)

2^31 - 1 = 2147483647 ( 10 chữ số )
số có 11 chữ số trở lên luôn >= 10^10 > 2^31: True


## 3. Kiến thức nền

### 3.1 Cắt chuỗi và đổi sang số nguyên

`num[i:j]` là đoạn từ chỉ số `i` tới `j − 1`. `int("0012")` cho `12` — **không** báo lỗi dù có số 0 đứng đầu; vì vậy ta phải **tự kiểm tra** quy tắc "không có số 0 thừa": đoạn có độ dài `> 1` mà ký tự đầu là `'0'` thì không hợp lệ.

In [2]:
num = "112358"
print(num[0:2], num[2:3], num[3:], int("0012"), int("0"), int("000"))


def has_leading_zero(piece):
    return len(piece) > 1 and piece[0] == "0"


print([has_leading_zero(p) for p in ("0", "00", "01", "10", "0123")])

11 2 358 12 0 0
[False, True, True, False, True]


### 3.2 Phép kiểm tra "một đoạn có xuất hiện đúng vị trí `pos`": `startswith`

`s.startswith(sub, pos)` kiểm tra `s[pos:pos+len(sub)] == sub`. Với bài này, sau khi có hai số đầu, số kế tiếp **đã biết** (bằng tổng) nên ta chỉ cần hỏi "chuỗi tại vị trí này có bắt đầu bằng chuỗi số đó không", không cần thử mọi độ dài.

In [3]:
s = "123456579"
print(s.startswith("456", 3), s.startswith("579", 6), s.startswith("57", 6), s.startswith("579", 5))

True True True False


### 3.3 Giới hạn 32-bit trong Python

Python có số nguyên cỡ tuỳ ý nên tổng `f[i] + f[i+1]` không tràn; đề lại yêu cầu mỗi số `< 2^31`. Ta phải **tự so sánh** với `2^31 − 1`. Hệ quả quan trọng: mỗi số tối đa **10 chữ số**, nên khi thử độ dài đoạn ta chỉ cần xét `≤ 10`.

### 3.4 Quay lui (backtracking) và cắt tỉa (pruning)

**Quay lui:** thử một lựa chọn, đi tiếp; nếu bế tắc thì **hoàn tác** lựa chọn rồi thử lựa chọn khác. **Cắt tỉa:** bỏ ngay những lựa chọn chắc chắn không dẫn tới nghiệm (ở đây: đoạn lớn hơn tổng hai số trước thì mọi đoạn dài hơn cũng lớn hơn → dừng).

### 3.5 Bài có nhiều đáp án: kiểm bằng hàm hợp lệ

Khi đề cho "bất kỳ đáp án đúng", test **không** nên viết `assert result == [11, 0, 11, 11]`. Ta viết hàm `is_valid_fib(num, seq)` kiểm tra mọi điều kiện của đề, và để kiểm tra **tồn tại**, dùng một oracle liệt kê **tất cả** đáp án (hoặc biết chắc không có).

In [4]:
def is_valid_fib(num, seq):
    """Kiểm tra seq có phải một cách tách hợp lệ của num theo ĐÚNG mọi điều kiện của đề không."""
    if not isinstance(seq, list) or len(seq) < 3:
        return False
    if any(not isinstance(x, int) or x < 0 or x > LIMIT for x in seq):
        return False
    if "".join(map(str, seq)) != num:                       # ghép lại phải ra đúng num (kéo theo: không có số 0 thừa, dùng hết chuỗi)
        return False
    return all(seq[i] + seq[i + 1] == seq[i + 2] for i in range(len(seq) - 2))


for seq in ([11, 0, 11, 11], [110, 1, 111], [11, 0, 11, 12], [1101111]):
    print(seq, "->", is_valid_fib("1101111", seq))
print(is_valid_fib("0123", [1, 2, 3]), is_valid_fib("0123", [0, 1, 2, 3]), is_valid_fib("0123", []))

[11, 0, 11, 11] -> True
[110, 1, 111] -> True
[11, 0, 11, 12] -> False
[1101111] -> False
False False False


Hàm trên dùng một mẹo gọn: **`"".join(map(str, seq)) == num`**. Vì `str(x)` của số nguyên không bao giờ có số 0 thừa (trừ `"0"`), nên điều kiện này đồng thời bảo đảm (a) ghép các đoạn lại ra đúng `num`, (b) không có đoạn nào có số 0 thừa, (c) dùng hết toàn bộ chuỗi.

## 4. Ý tưởng nền tảng / trực giác

Quan sát then chốt: **hai số đầu quyết định toàn bộ dãy**, vì số tiếp theo luôn là tổng hai số trước. Với `"123456579"`: nếu hai số đầu là `123` và `456` thì số kế là `579`, và chuỗi còn lại đúng là `"579"` → khớp, ta được `[123, 456, 579]`.

Vậy không cần thử mọi cách cắt (`2^(n−1)` cách); chỉ cần **chọn độ dài hai số đầu**, rồi **dựng** phần còn lại và so khớp với chuỗi:

1. Thử độ dài số thứ nhất `i = 1, 2, …` và số thứ hai `j = 1, 2, …` (mỗi số `< 2^31` nên `≤ 10` chữ số).
2. Với cặp `(a, b)` hợp lệ (không có số 0 thừa, `< 2^31`): đặt `c = a + b`, kiểm tra `num` có `str(c)` ở vị trí hiện tại không; nếu có thì tiếp tục với `(b, c)`.
3. Khi hết chuỗi mà đã có `≥ 3` số thì được đáp án.

Ví dụ `"1101111"`: cặp `(11, 0)` → `11` ✓ → `11` ✓ hết chuỗi: `[11, 0, 11, 11]`. Cặp `(110, 1)` → `111` ✓ hết: `[110, 1, 111]`. Có **hai** đáp án — đúng như đề nói.

In [5]:
def trace_pair(num, a_len, b_len):
    a_str, b_str = num[:a_len], num[a_len:a_len + b_len]
    seq = [int(a_str), int(b_str)]
    pos = a_len + b_len
    print(f"thử hai số đầu {seq}")
    while pos < len(num):
        c = seq[-1] + seq[-2]
        sc = str(c)
        ok = num.startswith(sc, pos)
        print(f"  cần {seq[-2]} + {seq[-1]} = {c}; tại vị trí {pos} chuỗi là {num[pos:pos + len(sc)]!r} -> {'khớp' if ok else 'KHÔNG khớp'}")
        if not ok:
            return None
        seq.append(c)
        pos += len(sc)
    print("  -> hết chuỗi:", seq)
    return seq


trace_pair("1101111", 2, 1)
trace_pair("1101111", 3, 1)
trace_pair("1101111", 1, 1)

thử hai số đầu [11, 0]
  cần 11 + 0 = 11; tại vị trí 3 chuỗi là '11' -> khớp
  cần 0 + 11 = 11; tại vị trí 5 chuỗi là '11' -> khớp
  -> hết chuỗi: [11, 0, 11, 11]
thử hai số đầu [110, 1]
  cần 110 + 1 = 111; tại vị trí 4 chuỗi là '111' -> khớp
  -> hết chuỗi: [110, 1, 111]
thử hai số đầu [1, 1]
  cần 1 + 1 = 2; tại vị trí 2 chuỗi là '0' -> KHÔNG khớp


## 5. Cách 1 — Brute Force: liệt kê mọi cách cắt rồi kiểm tra

**Ý tưởng:** chuỗi dài `n` có `n − 1` khe giữa các chữ số; chọn **tập con** các khe để cắt: `2^(n−1)` cách. Với mỗi cách cắt, đổi các đoạn sang số rồi kiểm tra bằng `is_valid_fib`. Chỉ dùng được với `n` nhỏ (≲ 18) nhưng **rõ ràng đúng theo đề**, và liệt kê được **mọi** đáp án.

In [6]:
def all_splits(num):
    """Mọi cách cắt num thành các đoạn (mỗi cách là list các chuỗi con)."""
    n = len(num)
    for mask in range(1 << (n - 1)):
        cuts = [i + 1 for i in range(n - 1) if mask >> i & 1]
        bounds = [0] + cuts + [n]
        yield [num[a:b] for a, b in zip(bounds, bounds[1:])]


def all_valid_brute(num):
    """Tập mọi dãy hợp lệ (dạng tuple) bằng cách vét cạn mọi cách cắt."""
    out = set()
    for pieces in all_splits(num):
        if any(has_leading_zero(p) for p in pieces) or len(pieces) < 3:
            continue
        seq = [int(p) for p in pieces]
        if is_valid_fib(num, seq):
            out.add(tuple(seq))
    return out


def split_brute(num):
    found = sorted(all_valid_brute(num))
    return list(found[0]) if found else []


for s in (ex1, ex2, ex3, "123456579", "112358"):
    print(f"{s!r:14}: mọi đáp án = {sorted(all_valid_brute(s))} -> trả {split_brute(s)}")
assert split_brute(ex2) == [] and split_brute(ex3) == []
assert all_valid_brute(ex1) == {(11, 0, 11, 11), (110, 1, 111)}

'1101111'     : mọi đáp án = [(11, 0, 11, 11), (110, 1, 111)] -> trả [11, 0, 11, 11]
'112358130'   : mọi đáp án = [] -> trả []
'0123'        : mọi đáp án = [] -> trả []
'123456579'   : mọi đáp án = [(123, 456, 579)] -> trả [123, 456, 579]
'112358'      : mọi đáp án = [(1, 1, 2, 3, 5, 8)] -> trả [1, 1, 2, 3, 5, 8]


**Độ phức tạp:** `2^(n−1)` cách cắt × `O(n)` kiểm tra = `O(n · 2^n)`; với `n = 200` là bất khả. Bộ nhớ `O(n)`.

**Nút thắt:** ta thử cả những cách cắt mà đoạn thứ ba **không** bằng tổng hai đoạn đầu. Biết hai số đầu là biết cả dãy.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Hai số đầu xác định mọi thứ.** Có `≤ 10` lựa chọn độ dài cho mỗi số (vì `< 2^31`), tức `≤ 100` cặp.
2. **Kiểm tra bằng dựng dãy.** Với một cặp, mỗi bước tính số kế tiếp bằng tổng rồi so với chuỗi (so chuỗi bằng `startswith` hoặc so số bằng `int`). Mỗi bước tiêu thụ ít nhất một chữ số nên tối đa `n` bước.
3. **Quay lui có cắt tỉa** là cách tổng quát hoá: duyệt từng đoạn kế tiếp, và khi đã có hai số, **ép** đoạn mới bằng tổng (cắt tỉa) — hiệu quả như cách (1) nhưng viết đệ quy.

Tổng thời gian cỡ `100 × n` thay vì `2^n`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** quay lui có cắt tỉa, **Cách 3** liệt kê hai số đầu rồi dựng dãy (so số), **Cách 4** liệt kê hai số đầu rồi dựng **chuỗi** để so với `num`.

### Cách 2 — Quay lui có cắt tỉa theo tổng

**Giải thích từng đoạn:**

1. `res` là dãy đang xây. `dfs(i)` xử lý từ chỉ số `i` của chuỗi.
2. Nếu `i == n`: thành công khi `len(res) >= 3`.
3. Thử các điểm kết thúc `j = i+1 … n`: đoạn `num[i:j]`.
   - Nếu đoạn dài `> 1` mà bắt đầu bằng `'0'` thì **dừng** (mọi đoạn dài hơn cũng có số 0 thừa).
   - Giá trị `x = int(num[i:j])`; nếu `x > 2^31 − 1` thì **dừng**.
   - Nếu đã có `≥ 2` số: `s = res[-1] + res[-2]`; nếu `x < s` bỏ qua (`continue`, thử đoạn dài hơn), nếu `x > s` **dừng** (mọi đoạn dài hơn còn lớn hơn).
   - Chọn `x`: `res.append(x)`, đệ quy `dfs(j)`; nếu thất bại thì `res.pop()` (hoàn tác).

In [7]:
def split_backtrack(num):
    n = len(num)
    res = []

    def dfs(i):
        if i == n:
            return len(res) >= 3
        for j in range(i + 1, n + 1):
            if j - i > 1 and num[i] == "0":              # số 0 thừa ở đầu: mọi đoạn dài hơn cũng sai
                break
            x = int(num[i:j])
            if x > LIMIT:                                # vượt 32-bit
                break
            if len(res) >= 2:
                s = res[-1] + res[-2]
                if x < s:
                    continue                             # còn quá nhỏ: thử đoạn dài hơn
                if x > s:
                    break                                # quá lớn: các đoạn dài hơn còn lớn hơn
            res.append(x)
            if dfs(j):
                return True
            res.pop()                                    # hoàn tác
        return False

    return res[:] if dfs(0) else []


for s in (ex1, ex2, ex3, "123456579"):
    r = split_backtrack(s)
    print(f"{s!r:14} -> {r}  hợp lệ: {is_valid_fib(s, r) if r else 'n/a (không có đáp án)'}")
assert is_valid_fib(ex1, split_backtrack(ex1)) and split_backtrack(ex2) == [] and split_backtrack(ex3) == []

'1101111'      -> [11, 0, 11, 11]  hợp lệ: True
'112358130'    -> []  hợp lệ: n/a (không có đáp án)
'0123'         -> []  hợp lệ: n/a (không có đáp án)
'123456579'    -> [123, 456, 579]  hợp lệ: True


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionBacktrack:
    def splitIntoFibonacci(self, num: str) -> List[int]:
        n = len(num)
        res = []
        limit = 2 ** 31 - 1

        def dfs(i: int) -> bool:
            if i == n:
                return len(res) >= 3
            for j in range(i + 1, n + 1):
                if j - i > 1 and num[i] == "0":
                    break
                x = int(num[i:j])
                if x > limit:
                    break
                if len(res) >= 2:
                    s = res[-1] + res[-2]
                    if x < s:
                        continue
                    if x > s:
                        break
                res.append(x)
                if dfs(j):
                    return True
                res.pop()
            return False

        return res if dfs(0) else []

**Độ phức tạp:** hai số đầu chọn trong `≤ 10 × 10` cách, sau đó mỗi bước chỉ còn **một** đoạn hợp lệ (đúng bằng tổng) nên rẽ nhánh gần như không có: tổng `O(100 · n)` thao tác (mỗi bước cắt/`int`). Bộ nhớ `O(n)` cho `res` và đệ quy.

### Cách 3 — Chỉ liệt kê hai số đầu, dựng dãy bằng tổng

**Giải thích từng đoạn:**

1. Duyệt độ dài số đầu `i ∈ [1, min(10, n)]` và số thứ hai `j ∈ [1, min(10, n − i)]`. Bỏ cặp có số 0 thừa hoặc `≥ 2^31`.
2. `seq = [a, b]`, `pos = i + j`. Lặp: `c = seq[-1] + seq[-2]`; nếu `c > 2^31 − 1` thì hỏng; nếu `num` không bắt đầu bằng `str(c)` tại `pos` thì hỏng; ngược lại `seq.append(c)`, `pos += len(str(c))`.
3. Hết chuỗi (`pos == n`) với `len(seq) ≥ 3` → trả `seq`.

In [9]:
def pair_candidates(num):
    """Sinh (a, b, vị trí bắt đầu phần còn lại) cho mọi cặp hai số đầu hợp lệ."""
    n = len(num)
    for i in range(1, min(10, n) + 1):
        if i > 1 and num[0] == "0":
            break
        a = int(num[:i])
        if a > LIMIT:
            break
        for j in range(1, min(10, n - i) + 1):
            if j > 1 and num[i] == "0":
                break
            b = int(num[i:i + j])
            if b > LIMIT:
                break
            yield a, b, i + j


def extend_from_pair(num, a, b, pos):
    seq = [a, b]
    n = len(num)
    while pos < n:
        c = seq[-1] + seq[-2]
        if c > LIMIT:
            return None
        sc = str(c)
        if not num.startswith(sc, pos):
            return None
        seq.append(c)
        pos += len(sc)
    return seq if len(seq) >= 3 else None


def all_valid_pairs(num):
    """Tập mọi dãy hợp lệ bằng cách liệt kê hai số đầu."""
    out = set()
    for a, b, pos in pair_candidates(num):
        seq = extend_from_pair(num, a, b, pos)
        if seq:
            out.add(tuple(seq))
    return out


def split_pairs(num):
    for a, b, pos in pair_candidates(num):
        seq = extend_from_pair(num, a, b, pos)
        if seq:
            return seq
    return []


for s in (ex1, ex2, ex3, "123456579"):
    r = split_pairs(s)
    print(f"{s!r:14} -> {r}")
    assert (not r and not all_valid_pairs(s)) or is_valid_fib(s, r)
assert all_valid_pairs(ex1) == {(11, 0, 11, 11), (110, 1, 111)}

'1101111'      -> [11, 0, 11, 11]
'112358130'    -> []
'0123'         -> []
'123456579'    -> [123, 456, 579]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionPairs:
    def splitIntoFibonacci(self, num: str) -> List[int]:
        n, limit = len(num), 2 ** 31 - 1
        for i in range(1, min(10, n) + 1):
            if i > 1 and num[0] == "0":
                break
            a = int(num[:i])
            if a > limit:
                break
            for j in range(1, min(10, n - i) + 1):
                if j > 1 and num[i] == "0":
                    break
                b = int(num[i:i + j])
                if b > limit:
                    break
                seq, pos = [a, b], i + j
                while pos < n:
                    c = seq[-1] + seq[-2]
                    sc = str(c)
                    if c > limit or not num.startswith(sc, pos):
                        break
                    seq.append(c)
                    pos += len(sc)
                else:
                    if len(seq) >= 3:
                        return seq
        return []

**Độ phức tạp:** `≤ 100` cặp × `O(n)` bước dựng = `O(100 · n)`; bộ nhớ `O(n)`. Không đệ quy, không hoàn tác.

### Cách 4 — Dựng chuỗi kỳ vọng rồi so sánh với `num`

**Ý tưởng khác:** với mỗi cặp hai số đầu, **dựng cả chuỗi** các chữ số của dãy bằng cách nối `str(a) + str(b) + str(a+b) + …` cho tới khi độ dài `≥ n`, rồi **so** với `num`. Nếu bằng nhau, dãy tương ứng là đáp án. Cách này chuyển bài toán thành "so sánh hai chuỗi" thay vì kiểm tra từng bước.

In [11]:
def split_build_string(num):
    n = len(num)
    for a, b, _ in pair_candidates(num):
        seq = [a, b]
        built = str(a) + str(b)
        ok = True
        while len(built) < n:
            c = seq[-1] + seq[-2]
            if c > LIMIT:
                ok = False
                break
            seq.append(c)
            built += str(c)
        if ok and built == num and len(seq) >= 3:
            return seq
    return []


for s in (ex1, ex2, ex3, "123456579", "0000"):
    r = split_build_string(s)
    print(f"{s!r:14} -> {r}")
    assert (not r and not all_valid_pairs(s)) or is_valid_fib(s, r)

'1101111'      -> [11, 0, 11, 11]
'112358130'    -> []
'0123'         -> []
'123456579'    -> [123, 456, 579]
'0000'         -> [0, 0, 0, 0]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionBuild:
    def splitIntoFibonacci(self, num: str) -> List[int]:
        n, limit = len(num), 2 ** 31 - 1
        for i in range(1, min(10, n) + 1):
            if i > 1 and num[0] == "0":
                break
            a = int(num[:i])
            if a > limit:
                break
            for j in range(1, min(10, n - i) + 1):
                if j > 1 and num[i] == "0":
                    break
                b = int(num[i:i + j])
                if b > limit:
                    break
                seq, built = [a, b], str(a) + str(b)
                while len(built) < n and seq[-1] + seq[-2] <= limit:
                    seq.append(seq[-1] + seq[-2])
                    built += str(seq[-1])
                if built == num and len(seq) >= 3:
                    return seq
        return []

**Độ phức tạp:** mỗi cặp dựng chuỗi dài `~n` (nối chuỗi `O(n)` tổng) nên `O(100 · n)`; bộ nhớ `O(n)`. Dễ viết nhưng cần cẩn thận kiểm `built == num` **sau cùng** (không phải kiểm tiền tố).

## 8. TRACE từng bước

**Trace Cách 3** với `num = "1101111"` (`n = 7`): duyệt hai số đầu.

| độ dài `(i, j)` | hai số đầu `(a, b)` | số kế `c = a + b` | khớp `num` tại vị trí? | kết quả |
|---|---|---|---|---|
| `(1, 1)` | `(1, 1)` | `2` | `num[2:3] = "0"` ≠ `"2"` | hỏng |
| `(1, 2)` | `(1, 10)` | `11` | `num[3:5] = "11"` ✓, vị trí 5 | tiếp: `c = 21`, `num[5:7] = "11"` ≠ `"21"` → hỏng |
| `(1, 3)` | `(1, 101)` | `102` | `num[4:7] = "111"` ≠ `"102"` | hỏng (các cặp `(1, j)` với `j ≥ 4` cũng hỏng) |
| `(2, 1)` | `(11, 0)` | `11` | `num[3:5] = "11"` ✓ | tiếp: `c = 11`, `num[5:7] = "11"` ✓ hết → `[11, 0, 11, 11]` (dừng ở đáp án đầu tiên; cặp `(3, 1) = (110, 1)` cũng cho `[110, 1, 111]`) |

Hàm trace in từng cặp hai số đầu được thử, dừng ở đáp án đầu tiên:

In [13]:
def trace_pairs(num):
    n = len(num)
    print(f"num = {num!r} (n = {n})")
    for a, b, pos in pair_candidates(num):
        seq = [a, b]
        why = None
        while pos < n:
            c = seq[-1] + seq[-2]
            sc = str(c)
            if c > LIMIT:
                why = f"{c} vượt 2^31-1"
                break
            if not num.startswith(sc, pos):
                why = f"cần {sc} tại vị trí {pos} nhưng thấy {num[pos:pos + len(sc)]!r}"
                break
            seq.append(c)
            pos += len(sc)
        if why:
            print(f"  hai số đầu ({a}, {b}): hỏng sau {seq}: {why}")
        elif len(seq) >= 3:
            print(f"  hai số đầu ({a}, {b}): THÀNH CÔNG -> {seq}")
            return seq
        else:
            print(f"  hai số đầu ({a}, {b}): hết chuỗi nhưng chỉ có {len(seq)} số")
    print("  không có đáp án")
    return []


assert is_valid_fib(ex1, trace_pairs(ex1))
print()
assert trace_pairs(ex2) == []
print()
assert trace_pairs(ex3) == []

num = '1101111' (n = 7)
  hai số đầu (1, 1): hỏng sau [1, 1]: cần 2 tại vị trí 2 nhưng thấy '0'
  hai số đầu (1, 10): hỏng sau [1, 10, 11]: cần 21 tại vị trí 5 nhưng thấy '11'
  hai số đầu (1, 101): hỏng sau [1, 101]: cần 102 tại vị trí 4 nhưng thấy '111'
  hai số đầu (1, 1011): hỏng sau [1, 1011]: cần 1012 tại vị trí 5 nhưng thấy '11'
  hai số đầu (1, 10111): hỏng sau [1, 10111]: cần 10112 tại vị trí 6 nhưng thấy '1'
  hai số đầu (1, 101111): hết chuỗi nhưng chỉ có 2 số
  hai số đầu (11, 0): THÀNH CÔNG -> [11, 0, 11, 11]

num = '112358130' (n = 9)
  hai số đầu (1, 1): hỏng sau [1, 1, 2, 3, 5, 8, 13]: cần 21 tại vị trí 8 nhưng thấy '0'
  hai số đầu (1, 12): hỏng sau [1, 12]: cần 13 tại vị trí 3 nhưng thấy '35'
  hai số đầu (1, 123): hỏng sau [1, 123]: cần 124 tại vị trí 4 nhưng thấy '581'
  hai số đầu (1, 1235): hỏng sau [1, 1235]: cần 1236 tại vị trí 5 nhưng thấy '8130'
  hai số đầu (1, 12358): hỏng sau [1, 12358]: cần 12359 tại vị trí 6 nhưng thấy '130'
  hai số đầu (1, 123581): hỏng

**Trace Cách 2 (quay lui)** cho `"123456579"` (cây quyết định rút gọn nhờ cắt tỉa): ở độ sâu 0 thử `1`, `12`, `123`...; ở độ sâu 1 thử `2`, `23`, `234`, ...; từ độ sâu 2 (có hai số) chỉ còn đúng một đoạn khả dĩ (bằng tổng) nên cây gần như là đường thẳng. Hàm trace in từng bước đệ quy, thụt lề theo độ sâu:

In [14]:
def trace_backtrack(num, max_lines=60):
    n = len(num)
    res = []
    lines = [0]

    def log(depth, msg):
        if lines[0] < max_lines:
            print("  " * depth + msg)
        elif lines[0] == max_lines:
            print("  ... (rút gọn)")
        lines[0] += 1

    def dfs(i):
        if i == n:
            ok = len(res) >= 3
            log(len(res), f"hết chuỗi với {res} -> {'thành công' if ok else 'chưa đủ 3 số'}")
            return ok
        for j in range(i + 1, n + 1):
            if j - i > 1 and num[i] == "0":
                log(len(res), f"đoạn {num[i:j]!r} có số 0 thừa -> dừng thử đoạn dài hơn")
                break
            x = int(num[i:j])
            if x > LIMIT:
                log(len(res), f"{x} > 2^31-1 -> dừng")
                break
            if len(res) >= 2:
                s = res[-1] + res[-2]
                if x < s:
                    log(len(res), f"thử {x} < tổng {s} -> thử đoạn dài hơn")
                    continue
                if x > s:
                    log(len(res), f"thử {x} > tổng {s} -> dừng (cắt tỉa)")
                    break
            res.append(x)
            log(len(res) - 1, f"chọn {x} -> dãy {res}")
            if dfs(j):
                return True
            res.pop()
        return False

    ok = dfs(0)
    print("=> kết quả:", res if ok else [])
    return res[:] if ok else []


r = trace_backtrack("123456579")
assert r == [123, 456, 579]

chọn 1 -> dãy [1]
  chọn 2 -> dãy [1, 2]
    chọn 3 -> dãy [1, 2, 3]
      thử 4 < tổng 5 -> thử đoạn dài hơn
      thử 45 > tổng 5 -> dừng (cắt tỉa)
    thử 34 > tổng 3 -> dừng (cắt tỉa)
  chọn 23 -> dãy [1, 23]
    thử 4 < tổng 24 -> thử đoạn dài hơn
    thử 45 > tổng 24 -> dừng (cắt tỉa)
  chọn 234 -> dãy [1, 234]
    thử 5 < tổng 235 -> thử đoạn dài hơn
    thử 56 < tổng 235 -> thử đoạn dài hơn
    thử 565 > tổng 235 -> dừng (cắt tỉa)
  chọn 2345 -> dãy [1, 2345]
    thử 6 < tổng 2346 -> thử đoạn dài hơn
    thử 65 < tổng 2346 -> thử đoạn dài hơn
    thử 657 < tổng 2346 -> thử đoạn dài hơn
    thử 6579 > tổng 2346 -> dừng (cắt tỉa)
  chọn 23456 -> dãy [1, 23456]
    thử 5 < tổng 23457 -> thử đoạn dài hơn
    thử 57 < tổng 23457 -> thử đoạn dài hơn
    thử 579 < tổng 23457 -> thử đoạn dài hơn
  chọn 234565 -> dãy [1, 234565]
    thử 7 < tổng 234566 -> thử đoạn dài hơn
    thử 79 < tổng 234566 -> thử đoạn dài hơn
  chọn 2345657 -> dãy [1, 2345657]
    thử 9 < tổng 2345658 -> thử đoạn

## 9. Code nộp LeetCode

Chọn **liệt kê hai số đầu rồi dựng dãy** (Cách 3): không đệ quy, độ phức tạp rõ ràng `O(100 · n)`. Giữ nguyên `class Solution`, `splitIntoFibonacci(self, num)`:

In [15]:
from typing import List


class Solution:
    def splitIntoFibonacci(self, num: str) -> List[int]:
        n, limit = len(num), 2 ** 31 - 1
        for i in range(1, min(10, n) + 1):                  # độ dài số đầu (<= 10 chữ số vì < 2^31)
            if i > 1 and num[0] == "0":                     # số 0 thừa ở đầu
                break
            a = int(num[:i])
            if a > limit:
                break
            for j in range(1, min(10, n - i) + 1):          # độ dài số thứ hai
                if j > 1 and num[i] == "0":
                    break
                b = int(num[i:i + j])
                if b > limit:
                    break
                seq, pos = [a, b], i + j
                while pos < n:                              # dựng phần còn lại bằng tổng
                    c = seq[-1] + seq[-2]
                    sc = str(c)
                    if c > limit or not num.startswith(sc, pos):
                        break
                    seq.append(c)
                    pos += len(sc)
                else:                                       # hết chuỗi mà không hỏng
                    if len(seq) >= 3:
                        return seq
        return []


for s in (ex1, ex2, ex3):
    r = Solution().splitIntoFibonacci(s)
    print(repr(s), "->", r)
assert is_valid_fib(ex1, Solution().splitIntoFibonacci(ex1))
assert Solution().splitIntoFibonacci(ex2) == [] and Solution().splitIntoFibonacci(ex3) == []
assert Solution().splitIntoFibonacci("123456579") == [123, 456, 579]

'1101111' -> [11, 0, 11, 11]
'112358130' -> []
'0123' -> []


## 10. Vì sao đúng

**(a) Hai số đầu xác định dãy.** Với điều kiện `f[i] + f[i+1] = f[i+2]`, biết `f0, f1` thì `f2 = f0 + f1`, `f3 = f1 + f2`, … Nên mọi dãy hợp lệ ứng với **đúng một** cặp `(f0, f1)`, và một cặp cho tối đa một dãy. Do đó liệt kê cặp là liệt kê **mọi** đáp án.

**(b) Giới hạn độ dài.** Mỗi `f[k] < 2^31` có không quá `10` chữ số (`2^31 − 1 = 2147483647`). Vì vậy độ dài của hai số đầu trong `num` là `≤ 10`: chỉ `≤ 100` cặp.

**(c) Kiểm tra từng cặp đúng.** Với cặp `(a, b)` ở các vị trí `[0, i)` và `[i, i+j)`, ta lặp: số kế tiếp **phải** là `c = a + b` (đề), nó chiếm đúng `len(str(c))` chữ số tiếp theo của `num` (biểu diễn thập phân không có số 0 thừa, vì `str(c)` không có số 0 đầu) — nên điều kiện cần và đủ để dãy khớp là `num` bắt đầu bằng `str(c)` ở vị trí hiện tại. Mỗi bước tiêu thụ `≥ 1` chữ số nên dừng sau `≤ n` bước. Khi tiêu thụ **hết** chuỗi mà không hỏng và có `≥ 3` số thì `seq` thoả mọi điều kiện.

**(d) Quy tắc số 0 thừa.** Một đoạn hợp lệ là `"0"` hoặc đoạn không bắt đầu bằng `'0'`. Với hai số đầu, ta bỏ đoạn dài `> 1` bắt đầu bằng `'0'` (và vì mọi đoạn dài hơn cùng điểm bắt đầu cũng có số 0 thừa nên `break`). Các số về sau do `str(c)` sinh ra nên không có số 0 thừa.

**(e) Cắt tỉa của quay lui đúng.** Khi đã có hai số, đoạn kế tiếp phải bằng `s = res[-1] + res[-2]`. Đoạn `x < s` thì thử dài hơn (`x` tăng); `x > s` thì mọi đoạn dài hơn còn lớn hơn (vì chữ số thêm vào làm số lớn hơn hoặc bằng, mà `num[i]` không phải `'0'`... trường hợp `num[i] == '0'` đã bị chặn ở dòng trước), nên dừng. Do đó không bỏ sót nghiệm.

Ta kiểm chứng bằng máy: các cách cho **cùng tập đáp án tồn tại** với oracle vét cạn mọi cách cắt, trên **mọi** chuỗi chữ số độ dài ≤ 7 (mục 12D), và trên chuỗi dài tới 200 (mục 12E–F) bằng hàm hợp lệ.

## 11. Time / Space Complexity

Đặt `n = len(num)` (`≤ 200`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi cách cắt | `O(n · 2^n)` | `O(n)` | oracle, `n ≲ 16` |
| 2. Quay lui cắt tỉa | `O(100 · n)` | `O(n)` | đệ quy, độ sâu `≤ n` |
| 3. Hai số đầu + dựng dãy | `O(100 · n)` | `O(n)` | **đáp án chính** |
| 4. Dựng chuỗi kỳ vọng | `O(100 · n)` | `O(n)` | so chuỗi cuối cùng |

Đo thực tế (độ dài 200; một ca có đáp án, một ca không):

In [16]:
def fib_string(a, b, max_len):
    """Nối các số của dãy bắt đầu (a, b) tới khi sắp vượt max_len hoặc vượt 2^31-1; trả (chuỗi, dãy)."""
    seq = [a, b]
    s = str(a) + str(b)
    while True:
        c = seq[-1] + seq[-2]
        if c > LIMIT or len(s) + len(str(c)) > max_len:
            break
        seq.append(c)
        s += str(c)
    return s, seq


s1, seq1 = fib_string(1, 2, 200)
print("chuỗi có đáp án: n =", len(s1), "| số phần tử", len(seq1), "| bắt đầu:", s1[:30], "...")
s2 = s1[:-1] + str((int(s1[-1]) + 1) % 10)                   # đổi chữ số cuối -> thường vô nghiệm
print("chuỗi vô nghiệm (đổi chữ số cuối): n =", len(s2))


def bench(fn, s, repeat=50):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(s)
        best = min(best, time.perf_counter() - t0)
    return best


for name, fn in [("backtrack", split_backtrack), ("pairs", split_pairs), ("build_string", split_build_string)]:
    print(f"{name:13s} có đáp án {bench(fn, s1) * 1e6:8.1f} µs | vô nghiệm {bench(fn, s2) * 1e6:8.1f} µs")
t0 = time.perf_counter()
split_brute(s1[:16])
print(f"brute n = 16: {(time.perf_counter() - t0) * 1000:.0f} ms (n = 200 là bất khả)")

chuỗi có đáp án: n = 195 | số phần tử 41 | bắt đầu: 123581321345589144233377610987 ...
chuỗi vô nghiệm (đổi chữ số cuối): n = 195
backtrack     có đáp án     44.9 µs | vô nghiệm    235.6 µs
pairs         có đáp án      9.7 µs | vô nghiệm     71.3 µs
build_string  có đáp án      9.4 µs | vô nghiệm    279.2 µs


brute n = 16: 233 ms (n = 200 là bất khả)


## 12. Bộ test

Vì bài có **nhiều đáp án đúng**, ta kiểm tra bằng **tính chất**: (1) nếu cài đặt trả dãy khác rỗng thì `is_valid_fib` phải `True`; (2) cài đặt trả `[]` **khi và chỉ khi** không tồn tại dãy hợp lệ (theo oracle liệt kê mọi dãy).

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi chuỗi chữ số ngắn, (E) ngẫu nhiên có seed (gieo dãy hợp lệ / làm hỏng), (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "backtrack": split_backtrack,
    "pairs": split_pairs,
    "build_string": split_build_string,
    "Solution": lambda s: Solution().splitIntoFibonacci(s),
    "SolutionBacktrack": lambda s: SolutionBacktrack().splitIntoFibonacci(s),
    "SolutionPairs": lambda s: SolutionPairs().splitIntoFibonacci(s),
    "SolutionBuild": lambda s: SolutionBuild().splitIntoFibonacci(s),
}


def check(num, exists=None, names=None):
    """exists=None: tính bằng oracle pair-enumeration (và brute nếu đủ ngắn)."""
    truth = all_valid_pairs(num)
    if len(num) <= 14:
        assert truth == all_valid_brute(num), ("hai oracle không khớp", num)
    if exists is not None:
        assert bool(truth) == exists, (num, truth, exists)
    for name in (names or IMPLS):
        got = IMPLS[name](num)
        if truth:
            assert got and is_valid_fib(num, got), (name, num, got, "phải có đáp án hợp lệ")
        else:
            assert got == [], (name, num, got, "phải là []")
    return truth


check(ex1, True)
check(ex2, False)
check(ex3, False)
assert is_valid_fib("123456579", Solution().splitIntoFibonacci("123456579"))
print("ví dụ 1, 2, 3: cả", len(IMPLS), "cài đặt trả dãy hợp lệ khi có, [] khi không có")

ví dụ 1, 2, 3: cả 7 cài đặt trả dãy hợp lệ khi có, [] khi không có


### B. Edge case

- Chuỗi ngắn hơn 3 chữ số (`"1"`, `"12"`) → `[]` (cần `≥ 3` số).
- `"112"` → `[1, 1, 2]`; `"101"` → `[1, 0, 1]`; `"000"` → `[0, 0, 0]`; `"0000"` → `[0, 0, 0, 0]`.
- Số 0 thừa: `"0123"` → `[]`; `"011"` → `[0, 1, 1]`; `"1011"` → `[1, 0, 1, 1]`.
- Số vượt 32-bit: dãy chứa `2147483648` bị cấm.
- Số lớn sát giới hạn: `[1073741823, 1073741824, 2147483647]` hợp lệ.
- Dãy dài (`112358132134…`).
- Nhiều đáp án (`"1101111"`, `"0000"`).

In [18]:
cases = {
    "1": False, "12": False, "112": True, "101": True, "000": True, "0000": True, "011": True,
    "0123": False, "1011": True, "01": False, "10": False, "1203": False,
    "123456579": True, "112358": True, "11235813": True, "1101111": True, "112358130": False,
    "199100199": True,
}
for num, want in cases.items():
    check(num, want)
# vượt 32-bit: 2147483647 + 1 = 2147483648 (không được dùng làm một số của dãy)
big_ok = "1073741823" + "1073741824" + "2147483647"
big_bad = "1073741824" + "1073741824" + "2147483648"
assert is_valid_fib(big_ok, [1073741823, 1073741824, 2147483647])
check(big_ok, True)
check(big_bad, False)
for name, fn in IMPLS.items():
    assert fn(big_bad) == [], name
# chuỗi dài hợp lệ: Fibonacci chuẩn
long_s, long_seq = fib_string(1, 1, 200)
check(long_s, True)
print(len(cases), "edge case +", 3, "ca 32-bit/dài: mọi cài đặt đúng; dãy dài có", len(long_seq), "số")

18 edge case + 3 ca 32-bit/dài: mọi cài đặt đúng; dãy dài có 42 số


### C. Phản ví dụ: cài đặt sai phải bị bắt

Mỗi bản sai phải bị **bắt** bằng một trong hai cách: trả dãy **không hợp lệ** (`is_valid_fib` sai), hoặc trả `[]` khi **có** đáp án (oracle), hoặc ngược lại trả dãy khi **không có**.

In [19]:
def pair_gen_flex(num, check_limit=True, check_zero=True, min_len=3, require_full=True, zero_piece_ok=True):
    """Bộ khung liệt kê hai số đầu với từng 'lỗi' bật/tắt được, để tạo các bản sai."""
    n = len(num)
    for i in range(1, n + 1):
        if check_zero and i > 1 and num[0] == "0":
            break
        a = int(num[:i])
        if check_limit and a > LIMIT:
            break
        if not zero_piece_ok and a == 0:
            continue
        for j in range(1, n - i + 1):
            if check_zero and j > 1 and num[i] == "0":
                break
            b = int(num[i:i + j])
            if check_limit and b > LIMIT:
                break
            if not zero_piece_ok and b == 0:
                continue
            seq, pos = [a, b], i + j
            while pos < n:
                c = seq[-1] + seq[-2]
                sc = str(c)
                if check_limit and c > LIMIT:
                    break
                if not num.startswith(sc, pos):
                    break
                seq.append(c)
                pos += len(sc)
            else:
                if len(seq) >= min_len:
                    return seq
            if not require_full and len(seq) >= min_len:
                return seq
    return []


bug_no_limit = lambda s: pair_gen_flex(s, check_limit=False)               # quên 32-bit
bug_leading_zero = lambda s: pair_gen_flex(s, check_zero=False)            # cho phép số 0 thừa
bug_len2 = lambda s: pair_gen_flex(s, min_len=2)                           # chấp nhận dãy 2 số
bug_prefix_only = lambda s: pair_gen_flex(s, require_full=False)           # không cần dùng hết chuỗi
bug_no_zero = lambda s: pair_gen_flex(s, zero_piece_ok=False)              # cấm luôn số 0 đơn lẻ

BUGS = {"quên 32-bit": bug_no_limit, "cho phép số 0 thừa": bug_leading_zero, "cho phép dãy 2 số": bug_len2,
        "không dùng hết chuỗi": bug_prefix_only, "cấm số 0 đơn lẻ": bug_no_zero}
rng = random.Random(1)
tests = ["".join(rng.choice("0123") for _ in range(rng.randint(2, 9))) for _ in range(1500)]
tests += ["0123", "1" * 5, "000", "1073741824" + "1073741824" + "2147483648", "12", "199100199", "112358130"]


def bug_detected(fn, s):
    got = fn(s)
    truth = all_valid_pairs(s)
    if got:
        return not is_valid_fib(s, got)             # trả dãy không hợp lệ
    return bool(truth)                              # trả [] nhưng có đáp án


for name, fn in BUGS.items():
    bad = next((s for s in tests if bug_detected(fn, s)), None)
    assert bad is not None, name
    print(f"{name:22s} sai trên {bad!r}: ra {fn(bad)}, tồn tại đáp án: {bool(all_valid_pairs(bad))}")

quên 32-bit            sai trên '107374182410737418242147483648': ra [1073741824, 1073741824, 2147483648], tồn tại đáp án: False
cho phép số 0 thừa     sai trên '20013': ra [2, 1, 3], tồn tại đáp án: False
cho phép dãy 2 số      sai trên '0203': ra [0, 203], tồn tại đáp án: False
không dùng hết chuỗi   sai trên '2023': ra [2, 0, 2], tồn tại đáp án: False
cấm số 0 đơn lẻ        sai trên '020302030': ra [], tồn tại đáp án: True


### D. Vét cạn mọi chuỗi chữ số ngắn

Với bảng chữ `{0, 1, 2, 3}` ta liệt kê **mọi** chuỗi độ dài `1..8` (`4 + 16 + … + 65 536 = 87 380` chuỗi) và với bảng chữ `{0, 1}` mọi chuỗi độ dài `1..14`. Với mỗi chuỗi: oracle vét cạn mọi cách cắt (`all_valid_brute`) phải **bằng** oracle hai-số-đầu (`all_valid_pairs`), và mọi cài đặt phải thoả tính chất ở mục 12A.

In [20]:
t0 = time.perf_counter()
cnt = 0
exist = 0
for alphabet, max_len in (("0123", 7), ("01", 12)):
    for L in range(1, max_len + 1):
        for t in product(alphabet, repeat=L):
            s = "".join(t)
            truth = all_valid_pairs(s)
            if L <= 11 or alphabet == "0123":
                assert truth == all_valid_brute(s), s
            for name in ("backtrack", "pairs", "build_string"):
                got = IMPLS[name](s)
                assert (not got and not truth) or (got and is_valid_fib(s, got) and tuple(got) in truth), (name, s, got)
            cnt += 1
            exist += bool(truth)
print(cnt, "chuỗi vét cạn; có đáp án:", exist, "; oracle brute == oracle hai-số-đầu; 3 cài đặt hợp lệ; mất", round(time.perf_counter() - t0, 1), "s")

30034 chuỗi vét cạn; có đáp án: 525 ; oracle brute == oracle hai-số-đầu; 3 cài đặt hợp lệ; mất 14.3 s


### E. Ngẫu nhiên có seed: gieo dãy hợp lệ, làm hỏng, chuỗi ngẫu nhiên

1. **Gieo:** chọn hai số đầu ngẫu nhiên (kích thước nhỏ, vừa, sát `2^31`), sinh dãy, nối thành chuỗi dài tới 200 (luôn có đáp án).
2. **Làm hỏng:** đổi một chữ số, hoặc cắt bớt, hoặc thêm một chữ số: so với oracle.
3. **Ngẫu nhiên thuần.**

In [21]:
def plant(rng, max_len=200):
    kind = rng.random()
    if kind < 0.4:
        a, b = rng.randint(0, 9), rng.randint(0, 9)
    elif kind < 0.8:
        a, b = rng.randint(0, 10 ** 4), rng.randint(0, 10 ** 4)
    else:
        a, b = rng.randint(0, LIMIT // 4), rng.randint(0, LIMIT // 4)
    s, seq = fib_string(a, b, rng.randint(3, max_len))
    return s, seq


rng = random.Random(20240842)
n_valid = n_broken = n_rand = n_has = 0
for _ in range(1500):
    s, seq = plant(rng)
    if len(seq) >= 3:
        check(s, True)                                        # gieo: chắc chắn có đáp án
        n_valid += 1
    pos = rng.randrange(len(s))
    mode = rng.random()
    if mode < 0.5:
        t = s[:pos] + str((int(s[pos]) + rng.randint(1, 9)) % 10) + s[pos + 1:]     # đổi một chữ số
    elif mode < 0.75:
        t = s[:pos] + s[pos + 1:] or "0"                      # xoá một chữ số
    else:
        t = s[:pos] + rng.choice("0123456789") + s[pos:]      # chèn một chữ số
    truth = check(t)
    n_broken += 1
    n_has += bool(truth)
    r = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 40)))
    check(r)
    n_rand += 1
print(f"gieo {n_valid} chuỗi hợp lệ, làm hỏng {n_broken} (còn đáp án: {n_has}), ngẫu nhiên {n_rand}: mọi cài đặt đúng tính chất")

gieo 1435 chuỗi hợp lệ, làm hỏng 1500 (còn đáp án: 4), ngẫu nhiên 1500: mọi cài đặt đúng tính chất


### F. Giới hạn lớn nhất

Chuỗi độ dài `200`: dãy Fibonacci dài nhất nối liền; chuỗi `200` chữ số `'0'` (cũng có đáp án `[0, 0, …, 0]`); `200` chữ số `'1'` (vô nghiệm?); số sát `2^31`; hai đáp án khác nhau cùng một chuỗi.

In [22]:
long_s, long_seq = fib_string(1, 1, 200)
cases = {
    "Fibonacci chuẩn tới 200": long_s,
    "200 chữ số 0": "0" * 200,
    "200 chữ số 1": "1" * 200,
    "dãy chẵn lẻ xen kẽ": fib_string(2, 3, 200)[0],
    "số lớn sát 2^31": fib_string(1000000000, 1000000000, 200)[0],
    "chuỗi tuần hoàn 12": "12" * 100,
}
for name, s in cases.items():
    truth = check(s)
    r = Solution().splitIntoFibonacci(s)
    print(f"{name:26s}: n={len(s):3d}, số đáp án tồn tại = {len(truth)}, trả về {'dãy dài ' + str(len(r)) if r else '[]'}")
assert Solution().splitIntoFibonacci("0" * 200) == [0] * 200
assert len(all_valid_pairs("0" * 200)) == 1
assert Solution().splitIntoFibonacci(long_s) and is_valid_fib(long_s, Solution().splitIntoFibonacci(long_s))

Fibonacci chuẩn tới 200   : n=196, số đáp án tồn tại = 1, trả về dãy dài 42
200 chữ số 0              : n=200, số đáp án tồn tại = 1, trả về dãy dài 200
200 chữ số 1              : n=200, số đáp án tồn tại = 0, trả về []
dãy chẵn lẻ xen kẽ        : n=194, số đáp án tồn tại = 1, trả về dãy dài 40
số lớn sát 2^31           : n= 30, số đáp án tồn tại = 1, trả về dãy dài 3
chuỗi tuần hoàn 12        : n=200, số đáp án tồn tại = 0, trả về []


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên giới hạn 32-bit

Python không tràn, nên `int("2147483648")` là một số hợp lệ trong Python — nhưng đề cấm số `≥ 2^31`.

In [23]:
s = "1073741824" + "1073741824" + "2147483648"
print("chuỗi:", s)
r = bug_no_limit(s)
print("bản quên 32-bit trả:", r, "| hợp lệ theo đề:", is_valid_fib(s, r))
print("lời giải đúng trả:", Solution().splitIntoFibonacci(s))
assert r and not is_valid_fib(s, r) and Solution().splitIntoFibonacci(s) == []

chuỗi: 107374182410737418242147483648
bản quên 32-bit trả: [1073741824, 1073741824, 2147483648] | hợp lệ theo đề: False
lời giải đúng trả: []


**Vì sao sai:** số `2147483648 = 2^31` không vừa số nguyên có dấu 32-bit. **Đúng:** `if x > 2**31 - 1: break` (và dừng thử đoạn dài hơn vì chỉ tăng).

### Lỗi 2: cho phép số 0 đứng đầu

`int("01")` bằng `1` nên nhiều người quên rằng đề **cấm** các đoạn như `"01"`. Ví dụ 3 (`"0123"`) chính là ca này.

In [24]:
print("bản cho phép số 0 thừa trên '0123':", bug_leading_zero("0123"), "| đúng:", Solution().splitIntoFibonacci("0123"))
print("ví dụ khác:", bug_leading_zero("011"), "->", is_valid_fib("011", bug_leading_zero("011")), "| đúng:", Solution().splitIntoFibonacci("011"))
assert bug_leading_zero("0123") and not is_valid_fib("0123", bug_leading_zero("0123"))

bản cho phép số 0 thừa trên '0123': [1, 2, 3] | đúng: []
ví dụ khác: [0, 1, 1] -> True | đúng: [0, 1, 1]


**Vì sao sai:** đoạn `"01"` có số 0 thừa. **Đúng:** đoạn dài `> 1` bắt đầu bằng `'0'` thì bỏ (và `break`), nhưng đoạn `"0"` đơn lẻ vẫn hợp lệ (`"011" → [0, 1, 1]`).

### Lỗi 3: chấp nhận dãy chỉ có 2 số

Đề yêu cầu `f.length >= 3`. `"12"` không thể tách; nhưng nếu không kiểm tra độ dài thì `[1, 2]` bị trả.

In [25]:
print("bản chấp nhận 2 số trên '12':", bug_len2("12"), "| đúng:", Solution().splitIntoFibonacci("12"))
assert bug_len2("12") == [1, 2] and Solution().splitIntoFibonacci("12") == []

bản chấp nhận 2 số trên '12': [1, 2] | đúng: []


**Vì sao sai:** dãy 2 số không thoả `len >= 3`. **Đúng:** chỉ trả khi `len(seq) >= 3`.

### Lỗi 4: không dùng hết chuỗi

Dãy tìm được phải **ghép lại ra đúng toàn bộ** `num`. Nếu chỉ kiểm tra phần đầu khớp rồi trả về, dãy bỏ sót các chữ số còn lại.

In [26]:
s = "11235x"[:5] + "9"                         # '112359': 1,1,2,3,5 rồi 9 không phải 8
r = bug_prefix_only(s)
print(repr(s), "-> bản không dùng hết chuỗi:", r, "| ghép lại:", "".join(map(str, r)) if r else None, "| hợp lệ:", is_valid_fib(s, r) if r else None)
print("lời giải đúng:", Solution().splitIntoFibonacci(s))
assert r and not is_valid_fib(s, r)

'112359' -> bản không dùng hết chuỗi: [1, 1, 2, 3, 5] | ghép lại: 11235 | hợp lệ: False
lời giải đúng: []


**Vì sao sai:** `[1, 1, 2, 3, 5]` ghép thành `"11235"`, thiếu chữ số `9`. **Đúng:** điều kiện thành công là `pos == n` (hoặc `i == n` trong quay lui) — tức đã dùng **hết** chuỗi.

### Lỗi 5: cấm luôn số 0 đơn lẻ

Quy tắc là "không có số 0 **thừa**"; số `0` chính nó hợp lệ (như `"000" → [0, 0, 0]`, `"101" → [1, 0, 1]`).

In [27]:
for s in ("000", "101", "1101111"):
    print(f"{s!r}: bản cấm số 0 -> {bug_no_zero(s)} | đúng -> {Solution().splitIntoFibonacci(s)}")
assert bug_no_zero("000") == [] and Solution().splitIntoFibonacci("000") == [0, 0, 0]

'000': bản cấm số 0 -> [] | đúng -> [0, 0, 0]
'101': bản cấm số 0 -> [] | đúng -> [1, 0, 1]
'1101111': bản cấm số 0 -> [110, 1, 111] | đúng -> [11, 0, 11, 11]


**Vì sao sai:** `"0"` là biểu diễn hợp lệ của số `0`. **Đúng:** chỉ chặn đoạn **dài hơn 1** bắt đầu bằng `'0'`.

### Lỗi 6 (bonus): so sánh test bằng `==` với một đáp án cố định

Bài có nhiều đáp án: `"1101111"` có cả `[11,0,11,11]` và `[110,1,111]`; test `assert result == [11, 0, 11, 11]` có thể **sai** với một lời giải đúng.

In [28]:
r_a = Solution().splitIntoFibonacci("1101111")
r_b = SolutionBacktrack().splitIntoFibonacci("1101111")
print("Solution:", r_a, "| SolutionBacktrack:", r_b, "| bằng nhau?", r_a == r_b)
assert is_valid_fib("1101111", r_a) and is_valid_fib("1101111", r_b)
print("mọi đáp án tồn tại:", sorted(all_valid_pairs("1101111")))

Solution: [11, 0, 11, 11] | SolutionBacktrack: [11, 0, 11, 11] | bằng nhau? True
mọi đáp án tồn tại: [(11, 0, 11, 11), (110, 1, 111)]


**Vì sao sai:** thứ tự thử khác nhau cho đáp án đầu tiên khác nhau, cả hai đều đúng. **Đúng:** kiểm bằng hàm hợp lệ `is_valid_fib`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tách chuỗi thành các phần thoả quan hệ truyền (kiểu `f[i] + f[i+1] = f[i+2]`)" → **quay lui có cắt tỉa**, hoặc **liệt kê điều kiện khởi đầu** (hai số đầu) rồi suy ra mọi thứ.

**Mẹo:**

1. Tìm **bậc tự do** của bài: ở đây chỉ hai số đầu; cả dãy còn lại bị ép.
2. Dùng **giới hạn giá trị** để chặn số lựa chọn (mỗi số `< 2^31` → `≤ 10` chữ số).
3. **Số 0 đứng đầu** luôn là bẫy: chặn đoạn dài `> 1` bắt đầu bằng `'0'`, cho phép `"0"` đơn lẻ.
4. Khi kiểm tra dãy khớp chuỗi: `str.startswith(sub, pos)` thay cho cắt lát nhiều lần.
5. Bài **nhiều đáp án**: viết hàm kiểm tra hợp lệ; kiểm "có / không có đáp án" bằng oracle liệt kê **mọi** đáp án.
6. Python không tràn số: nhớ tự kiểm `≤ 2**31 − 1`.

**Họ bài cùng pattern:** Additive Number (306, gần như cùng bài nhưng không giới hạn kích thước), Restore IP Addresses (93), Palindrome Partitioning (131), Word Break II (140), Split a String Into the Max Number of Unique Substrings (1593).

## 15. Biến thể

1. **Additive Number (306):** không giới hạn 32-bit (dùng số nguyên lớn của Python) và chỉ trả `True/False`.
2. **Đếm số cách tách hợp lệ** (bài tập 5).
3. **Dãy kiểu Fibonacci tổng quát** `f[i+2] = x·f[i+1] + y·f[i]`.
4. **Trả về dãy dài nhất / ngắn nhất** nếu có nhiều đáp án (độ dài khác nhau).
5. **Mở rộng:** cho phép đảo ngược thứ tự, hoặc cho phép số âm.

Cài đặt biến thể 1 (Additive Number, không giới hạn) và đối chiếu với oracle:

In [29]:
def additive_number(num):
    """LeetCode 306: dãy cộng tính (>= 3 số, không số 0 thừa), không giới hạn 32-bit (Python cho số lớn)."""
    n = len(num)
    for i in range(1, n):
        if i > 1 and num[0] == "0":
            break
        a = int(num[:i])
        for j in range(1, n - i):
            if j > 1 and num[i] == "0":
                break
            b = int(num[i:i + j])
            seq, pos = [a, b], i + j
            while pos < n:
                c = seq[-1] + seq[-2]
                sc = str(c)
                if not num.startswith(sc, pos):
                    break
                seq.append(c)
                pos += len(sc)
            else:
                return True
    return False


print(additive_number("112358"), additive_number("199100199"), additive_number("1023"), additive_number("101"))
assert additive_number("112358") and additive_number("199100199") and not additive_number("1023") and additive_number("101")
# số lớn: có đáp án 306 nhưng KHÔNG có đáp án của bài 842 (vì vượt 2^31)
s = "9" * 12 + "1" + "9" * 12 + "0"[:0]
big = str(10 ** 11) + str(10 ** 11) + str(2 * 10 ** 11)
print("306:", additive_number(big), "| 842:", Solution().splitIntoFibonacci(big))
assert additive_number(big) and Solution().splitIntoFibonacci(big) == []
# nếu có dãy 32-bit hợp lệ thì additive_number đúng
rng = random.Random(15)
for _ in range(300):
    s_, seq = plant(rng, 60)
    if len(seq) >= 3:
        assert additive_number(s_)
print("biến thể 306: nhất quán với dãy gieo")

True True False True
306: True | 842: []
biến thể 306: nhất quán với dãy gieo


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cách cắt | Liệt kê `2^(n−1)` cách cắt, kiểm hợp lệ | `O(n·2^n)` | `O(n)` | Oracle; `n ≲ 16` |
| 2. Quay lui cắt tỉa | Chọn đoạn kế tiếp, ép bằng tổng, hoàn tác | `O(100·n)` | `O(n)` | Tổng quát hoá (bài tách chuỗi khác) |
| 3. Hai số đầu + dựng dãy | Liệt kê `≤ 100` cặp, dãy còn lại suy ra | `O(100·n)` | `O(n)` | **Đáp án chính**; không đệ quy |
| 4. Dựng chuỗi kỳ vọng | Nối `str(a)+str(b)+str(a+b)+…` rồi so `num` | `O(100·n)` | `O(n)` | Gọn; kiểm một lần cuối |

Cả bốn trả dãy hợp lệ khi có và `[]` khi không có, khớp oracle trên 21 844 + 8 190 chuỗi vét cạn và hàng nghìn chuỗi ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [30]:
print(is_valid_fib("112358", [1, 1, 2, 3, 5, 8]), is_valid_fib("112358", [11, 2, 3, 5, 8]), is_valid_fib("0", [0]))
print(Solution().splitIntoFibonacci("0000"), Solution().splitIntoFibonacci("12"), Solution().splitIntoFibonacci("101"))
print(sorted(all_valid_pairs("1203")))

True False False
[0, 0, 0, 0] [] [1, 0, 1]
[]


### Bài 2 (tính tay)

`num = "112358"`.

- (a) Liệt kê hai số đầu có thể và dãy suy ra cho từng cặp.
- (b) Có bao nhiêu dãy hợp lệ?
- (c) Với `num = "1203"`: vì sao không có dãy nào? Chỉ ra lý do cho từng cặp hai số đầu (và nhớ rằng `"1011"` thì **có** đáp án — hãy tìm).

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm kiểm tra tính hợp lệ:

In [31]:
def is_valid_todo(num, seq):
    # TODO 1: len(seq) < 3 -> False
    # TODO 2: mỗi số phải trong [0, 2**31 - 1]
    # TODO 3: ghép "".join(map(str, seq)) phải bằng num
    # TODO 4: mọi i: seq[i] + seq[i+1] == seq[i+2]
    return None


# Khi làm xong, bỏ comment để kiểm tra:
# assert is_valid_todo("1101111", [11, 0, 11, 11]) is True and is_valid_todo("0123", [1, 2, 3]) is False
# assert is_valid_todo("112", [1, 1]) is False and is_valid_todo("1" * 3, [1, 1, 1]) is False

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` theo ý tưởng "hai số đầu rồi dựng dãy" (hoặc quay lui):

In [32]:
class MySolution:
    def splitIntoFibonacci(self, num):
        pass            # TODO


# Khi làm xong (kiểm bằng hàm hợp lệ):
# for s in ("1101111", "123456579", "112358"):
#     r = MySolution().splitIntoFibonacci(s)
#     assert r and is_valid_fib(s, r)
# assert MySolution().splitIntoFibonacci("0123") == [] and MySolution().splitIntoFibonacci("112358130") == []

### Bài 5 (mini challenge): đếm số cách tách hợp lệ

Viết `count_splits(num)` trả về **số dãy hợp lệ khác nhau** (theo đề 842, tức `≥ 3` số, `< 2^31`, không số 0 thừa) tách được từ `num`. Ví dụ `"1101111"` có `2`; `"0000"` có `1`. Gợi ý: số dãy bằng số **cặp hai số đầu** cho dãy khớp (mỗi cặp một dãy). Kiểm bằng oracle vét cạn mọi cách cắt.

In [33]:
def count_splits(num):
    pass            # TODO


# Khi làm xong:
# assert count_splits("1101111") == 2 and count_splits("0000") == 1 and count_splits("112358130") == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `is_valid_fib` yêu cầu `len ≥ 3`, mọi số trong khoảng, ghép lại bằng `num`, và quan hệ tổng.
- Mức 2: `[11,2,3,5,8]`: `11 + 2 = 13 ≠ 3` → sai; `is_valid_fib("0", [0])` sai vì `len < 3`.
- Mức 3: `True False False`; `[0,0,0,0]`, `[]`, `[1,0,1]`; với `"1203"` mọi cặp hai số đầu đều hỏng nên tập rỗng.

**Bài 2**

- Mức 1: hai số đầu của `"112358"` là các tiền tố.
- Mức 2: `(1,1)` → `2,3,5,8` khớp hết; `(1,12)`: `13` không khớp; `(11,2)`: `13` không khớp.
- Mức 3: (b) đúng `1` dãy `[1,1,2,3,5,8]`. (c) `"1203"`: `(1,2)` → `3` nhưng tại vị trí 2 là `'0'`; `(1,20)` → `21` không khớp; `(12,0)` → `12` không khớp `'3'`...; tất cả cặp đều hỏng. Còn `"1011"`: `(1,0)` → `1` ✓ rồi `1` ✓ → `[1,0,1,1]`.

**Bài 3**

- Mức 1: bốn điều kiện liên tiếp.
- Mức 2: `all(0 <= x <= 2**31 - 1 for x in seq)`.
- Mức 3: `"".join(map(str, seq)) == num` bao hàm cả "không số 0 thừa".

**Bài 4**

- Mức 1: hai vòng `for` độ dài hai số đầu (`≤ 10`).
- Mức 2: dựng dãy với `startswith`.
- Mức 3: xem mục 9.

**Bài 5**

- Mức 1: dãy tương ứng một-một với cặp hai số đầu.
- Mức 2: đếm cặp mà `extend_from_pair` thành công.
- Mức 3: hoặc `len(all_valid_pairs(num))`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [34]:
assert (is_valid_fib("112358", [1, 1, 2, 3, 5, 8]), is_valid_fib("112358", [11, 2, 3, 5, 8]), is_valid_fib("0", [0])) == (True, False, False)
assert Solution().splitIntoFibonacci("0000") == [0, 0, 0, 0] and Solution().splitIntoFibonacci("12") == []
assert Solution().splitIntoFibonacci("101") == [1, 0, 1]
print("all_valid_pairs('1203') =", sorted(all_valid_pairs("1203")))
assert all_valid_pairs("1203") == all_valid_brute("1203") == set()
print("Bài 1 OK")

all_valid_pairs('1203') = []
Bài 1 OK


### Bài 2

(a) Các cặp hai số đầu đáng thử: `(1,1)` → dãy `1,1,2,3,5,8` khớp hết; `(1,12)` → `13` không khớp; `(11,2)` → `13` không khớp. (b) Đúng `1` dãy `[1,1,2,3,5,8]`. (c) `"1203"`: `(1,2)` → `3` nhưng vị trí 2 là `'0'`; `(1,20)` → `21` không khớp; `(12,0)` → `12` không khớp `'3'`; `(120,3)`: hết chuỗi chỉ có 2 số; → không có dãy. Còn `"1011"` thì **có** `[1,0,1,1]`.

In [35]:
assert all_valid_pairs("112358") == {(1, 1, 2, 3, 5, 8)}
assert extend_from_pair("112358", 1, 12, 3) is None and extend_from_pair("112358", 11, 2, 3) is None
assert all_valid_pairs("1011") == {(1, 0, 1, 1)} == all_valid_brute("1011")
assert all_valid_pairs("1203") == all_valid_brute("1203") == set()
for a_, b_, pos_ in pair_candidates("1203"):
    assert extend_from_pair("1203", a_, b_, pos_) is None
print("Bài 2 OK: '112358' có", len(all_valid_pairs("112358")), "dãy; '1011' có", sorted(all_valid_pairs("1011")), "; '1203' không có")

Bài 2 OK: '112358' có 1 dãy; '1011' có [(1, 0, 1, 1)] ; '1203' không có


### Bài 3

In [36]:
def is_valid_answer(num, seq):
    if len(seq) < 3:
        return False
    if any(x < 0 or x > 2 ** 31 - 1 for x in seq):
        return False
    if "".join(map(str, seq)) != num:
        return False
    return all(seq[i] + seq[i + 1] == seq[i + 2] for i in range(len(seq) - 2))


assert is_valid_answer("1101111", [11, 0, 11, 11]) is True and is_valid_answer("0123", [1, 2, 3]) is False
assert is_valid_answer("112", [1, 1]) is False and is_valid_answer("111", [1, 1, 1]) is False
rng = random.Random(3)
for _ in range(500):
    s_ = "".join(rng.choice("0123") for _ in range(rng.randint(1, 8)))
    seq = [int(p) for p in rng.choice(list(all_splits(s_)))] if True else []
    assert is_valid_answer(s_, seq) == is_valid_fib(s_, seq)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def splitIntoFibonacci(self, num):
        n = len(num)
        limit = 2 ** 31 - 1
        for i in range(1, min(10, n) + 1):
            if i > 1 and num[0] == "0":
                break
            a = int(num[:i])
            if a > limit:
                break
            for j in range(1, min(10, n - i) + 1):
                if j > 1 and num[i] == "0":
                    break
                b = int(num[i:i + j])
                if b > limit:
                    break
                seq, built = [a, b], str(a) + str(b)
                while len(built) < n and seq[-1] + seq[-2] <= limit:
                    seq.append(seq[-1] + seq[-2])
                    built += str(seq[-1])
                if built == num and len(seq) >= 3:
                    return seq
        return []


for s in ("1101111", "123456579", "112358"):
    r = MySolutionAnswer().splitIntoFibonacci(s)
    assert r and is_valid_fib(s, r)
assert MySolutionAnswer().splitIntoFibonacci("0123") == [] and MySolutionAnswer().splitIntoFibonacci("112358130") == []
rng = random.Random(4)
for _ in range(500):
    s = "".join(rng.choice("0123") for _ in range(rng.randint(1, 10)))
    r = MySolutionAnswer().splitIntoFibonacci(s)
    assert (not r and not all_valid_pairs(s)) or (r and is_valid_fib(s, r))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def count_splits_answer(num):
    return sum(1 for a, b, pos in pair_candidates(num) if extend_from_pair(num, a, b, pos))


assert count_splits_answer("1101111") == 2 and count_splits_answer("0000") == 1 and count_splits_answer("112358130") == 0
rng = random.Random(5)
for _ in range(1000):
    s = "".join(rng.choice("0123") for _ in range(rng.randint(1, 12)))
    assert count_splits_answer(s) == len(all_valid_brute(s)) == len(all_valid_pairs(s)), s
print("Bài 5 OK: khớp oracle vét cạn mọi cách cắt trên 1000 chuỗi")

Bài 5 OK: khớp oracle vét cạn mọi cách cắt trên 1000 chuỗi


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (liệt kê hai số đầu, dựng dãy bằng tổng):**

```python
class Solution:
    def splitIntoFibonacci(self, num: str) -> List[int]:
        n, limit = len(num), 2 ** 31 - 1
        for i in range(1, min(10, n) + 1):
            if i > 1 and num[0] == "0": break
            a = int(num[:i])
            if a > limit: break
            for j in range(1, min(10, n - i) + 1):
                if j > 1 and num[i] == "0": break
                b = int(num[i:i + j])
                if b > limit: break
                seq, pos = [a, b], i + j
                while pos < n:
                    c = seq[-1] + seq[-2]
                    sc = str(c)
                    if c > limit or not num.startswith(sc, pos): break
                    seq.append(c); pos += len(sc)
                else:
                    if len(seq) >= 3: return seq
        return []
```

**Checklist trước khi submit:**

- [ ] Mỗi số `≤ 2**31 − 1` (kiểm cả số được suy ra bằng tổng)?
- [ ] Không có số 0 thừa: đoạn dài `> 1` bắt đầu bằng `'0'` bị loại; `"0"` đơn lẻ vẫn được?
- [ ] `len(seq) ≥ 3` và đã **dùng hết** chuỗi (`pos == n`)?
- [ ] Hoàn tác (`pop`) nếu dùng quay lui; cắt tỉa `x > s → break`?
- [ ] Test bằng **hàm kiểm tra hợp lệ**, không so cứng với một đáp án (bài nhiều đáp án)?
- [ ] Đã thử `"1101111"`, `"112358130"` (`[]`), `"0123"` (`[]`), `"000"`, số sát `2^31`?
- [ ] Đúng tên `class Solution`, `splitIntoFibonacci(self, num)`, không in debug.